# SELIX — Prova Formal Lean 4

Este notebook instala o Lean 4 e verifica formalmente que a **Selic ótima = 9,25%**.

> ⏱️ A instalação do Lean 4 leva ~5 minutos. Execute célula por célula.

[![GitHub](https://img.shields.io/badge/SELIX-GitHub-black)](https://github.com/scoobiii/selix)

In [ ]:
%%bash
set -e
echo '>>> Instalando elan (gerenciador do Lean 4)...'
curl -sSf https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh | sh -s -- -y --default-toolchain leanprover/lean4:stable 2>&1
export PATH=$HOME/.elan/bin:$PATH
lean --version
echo '>>> Lean 4 instalado com sucesso.'

In [ ]:
lean_code = '''
-- SELIX_simple.lean
-- Prova formal: Selic ótima = 9.25% dado inflação = 4.48%
def inflacao   : Float := 4.48
def teto_juro  : Float := 5.00
def roe_medio  : Float := 18.5
def media_glob : Float := 10.67
def step_copom : Float := 0.25
def teto1 : Float := inflacao + teto_juro
def teto2 : Float := roe_medio * 0.95
def teto3 : Float := media_glob
def min2 (a b : Float) : Float := if a < b then a else b
def min_tetos : Float := min2 teto1 (min2 teto2 teto3)
def selix_star : Float := (Int.floor (min_tetos / step_copom)).toFloat * step_copom
#eval (teto1 < teto2 && teto1 < teto3)
#eval (selix_star >= 6.48 && selix_star <= teto1)
#eval (9.50 > teto1)
#eval (14.50 > teto1)
#eval selix_star
'''
with open('/root/SELIX_simple.lean', 'w') as f:
    f.write(lean_code)
print('Arquivo criado')

In [ ]:
import subprocess, os
env = os.environ.copy()
env['PATH'] = os.path.expanduser('~/.elan/bin') + ':' + env['PATH']
result = subprocess.run(['lean', '/root/SELIX_simple.lean'], capture_output=True, text=True, env=env)
print('=== STDOUT ===')
print(result.stdout)
if result.stderr:
    print('=== STDERR ===')
    print(result.stderr)

In [ ]:
lines = result.stdout.strip().split('\n')
print('=' * 55)
print('  SELIX — Resultados Lean 4')
print('=' * 55)
for i, line in enumerate(lines):
    val = line.strip()
    if val in ('true', 'false'):
        print(f'  Teorema {i+1:<20} {"✅ OK" if val == "true" else "❌ FALHA"}')
    else:
        print(f'  SELIX ótimo: {val}%')
print('=' * 55)